In [1]:
import os 
from crewai import Crew, Agent, Task, Process, LLM
from dotenv import load_dotenv 
load_dotenv() 

llm = LLM(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

### **Tasks Guardrails**

In [2]:
def validate_summary_length(task_output):
    try:
        print("Validating summary length")
        task_str_output = str(task_output)
        total_words = len(task_str_output.split())

        print(f"Word Count: {total_words}")

        if total_words > 150:
            print("Summary exceeds 150 words")
            return (False, f"Summary exceeds 150 words. Word count: {total_words}")
        
        if total_words == 0:
            print("Summary is empty")
            return (False, "Generated summary is empty.")
        
        print("Summary is valid")
        return (True, task_output)

    except Exception as e:
        print("Validation system error")
        return (False, f"Validation system error: {str(e)}")

In [3]:
from crewai import Task, Agent

summary_agent = Agent(
    role="Summary Agent",
    goal="Summarize the research paper 'Convolutional Neural Networks' in 150 words.",
    backstory="You are a specialized agent that summarizes research papers.",
    verbose=True,
    llm=llm
)

summary_task = Task(
    description="Summarize a research paper in 150 words.",
    expected_output="A concise research summary 150 words.",
    agent=summary_agent,
    guardrail=validate_summary_length,

    max_retries=3
)

In [4]:
from crewai import Crew

summary_crew = Crew(
    agents=[summary_agent],
    tasks=[summary_task],
    verbose=True
)

result = summary_crew.kickoff()

┌────────────────────────── Crew Execution Started ───────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: ec17ef1d-ddb6-4624-a20d-527b340fe4d6                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

🚀 Crew: crew
└── 📋 Task: 0e6e7835-488c-4591-9d2a-a5a933119a63
    Status: Executing Task...
┌───────────────────────────── 🤖 Agent Started ──────────────────────────────┐
│                                                                             │
│  Agent: Summary Agent     

### Ask the agent to write a 200 word summary instead and notice the guardrail output:

In [5]:
from crewai import Task, Agent

summary_agent = Agent(
    role="Summary Agent",
    goal="Summarize the research paper 'Convolutional Neural Networks' in 200 words.",
    backstory="You are a specialized agent that summarizes research papers.",
    verbose=True,
    llm=llm
)

summary_task = Task(
    description="Summarize a research paper in 200 words.",
    expected_output="A concise research summary 200 words.",
    agent=summary_agent,
    guardrail=validate_summary_length,
    max_retries=3
)

In [6]:
from crewai import Crew

summary_crew = Crew(
    agents=[summary_agent],
    tasks=[summary_task],
    verbose=True
)

result = summary_crew.kickoff()

┌────────────────────────── Crew Execution Started ───────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: 95e4ea0f-7b98-40c6-908b-e57face05537                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

🚀 Crew: crew
└── 📋 Task: 9ad40291-0333-4ec0-80db-f6760be0b6e7
    Status: Executing Task...
┌───────────────────────────── 🤖 Agent Started ──────────────────────────────┐
│                                                                             │
│  Agent: Summary Agent     

Exception: Task failed guardrail validation after 3 retries. Last error: Summary exceeds 150 words. Word count: 184

### **Tasks Guardrails with Pydantic**

In [7]:
from pydantic import BaseModel

class ResearchReport(BaseModel):
    """Represents a structured research report"""
    title: str
    summary: str
    key_findings: list[str]

In [10]:
import json 
from typing import List, Any, Tuple 

def validate_json_report(result: str) -> Tuple[bool, Any]:
    """Ensures AI-generated output is valid JSON with required fields."""
    try:
        # Parse JSON output
        data = json.loads(result.pydantic.model_dump_json())

        # Check required fields
        if 'title' not in data or 'summary' not in data or 'key_findings' not in data:
            return (False, "Missing required fields: title, summary, or key_findings.")
        return (True, result) 
    except json.JSONDecodeError:
        return (False, "Invalid JSON format. Please ensure correct syntax.")

In [11]:
from crewai import Agent

# Create the AI Agent
research_report_agent = Agent(
    role="Research Analyst",
    goal="Generate structured JSON reports for research papers",
    backstory="You are an expert in technical writing and structured reporting.",
    verbose=False,
    llm=llm
)

In [12]:
from crewai import Task

research_report_task = Task(
    description="Generate a structured research report in valid JSON format.",
    expected_output="A valid JSON object containing 'title', 'summary', and 'key_findings'.",
    agent=research_report_agent,
    output_pydantic=ResearchReport,  # Ensures structured output
    guardrail=validate_json_report,  # Validate output before passing to next step
    max_retries=3  # Allow up to 3 retries if validation fails
)

In [13]:
from crewai import Crew

research_crew = Crew(
    agents=[research_report_agent],
    tasks=[research_report_task],
    verbose=True  # Display execution details
)

In [14]:
result = research_crew.kickoff()

# Display the validated JSON output
print("Final Research Report:", result.pydantic)

┌────────────────────────── Crew Execution Started ───────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: 1bbcbd2f-8297-4780-9f84-d5e083f54a64                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

🚀 Crew: crew
└── 📋 Task: 4f2a97b7-9916-4185-be7e-81614119a1fd
    Status: Executing Task...
┌───────────────────────────── 🤖 Agent Started ──────────────────────────────┐
│                                                                             │
│  Agent: Research Analyst  

### **Getting Structured Consistent Outputs from Tasks**

In [15]:
from pydantic import BaseModel

class ResearchFindings(BaseModel):
    """Structured research report output"""
    title: str
    key_findings: list[str]

class AnalysisSummary(BaseModel):
    """Structured summary of research findings"""
    insights: list[str]
    key_takeaways: str

In [16]:
from crewai import Agent, LLM

# AI Agents
research_agent = Agent(
    role="AI Researcher",
    goal="Find and summarize the latest AI advancements",
    backstory="You are an expert AI researcher who stays up to date with the latest innovations.",
    verbose=True,
    llm=llm
)

analysis_agent = Agent(
    role="AI Analyst",
    goal="Analyze AI research findings and extract key insights",
    backstory="You are a data analyst who extracts valuable insights from research data.",
    verbose=True,
    llm=llm
)

writer_agent = Agent(
    role="Tech Writer",
    goal="Write a well-structured blog post on AI trends",
    backstory="You are a technology writer skilled at transforming complex AI research into readable content.",
    verbose=True,
    llm=llm
)

In [17]:
from crewai import Task

# Step 1: Research Task
research_task = Task(
    description="Find and summarize the latest AI advancements",
    expected_output="A structured list of recent AI breakthroughs",
    agent=research_agent,
    output_pydantic=ResearchFindings  # Structured output
)

# Step 2: Analysis Task (References Research Task Output)
analysis_task = Task(
    description="Analyze AI research findings and extract key insights",
    expected_output="A structured summary with key takeaways",
    agent=analysis_agent,
    output_pydantic=AnalysisSummary,
    context=[research_task]  # Receives output from research_task
)

# Step 3: Blog Writing Task (References Both Research and Analysis)
blog_writing_task = Task(
    description="Write a detailed blog post about AI trends",
    expected_output="A well-structured blog post",
    agent=writer_agent,
    context=[research_task, analysis_task]  # Uses both research and analysis outputs
)

In [18]:
from crewai import Crew

ai_research_crew = Crew(
    agents=[research_agent, analysis_agent, writer_agent],
    tasks=[research_task, analysis_task, blog_writing_task],
    verbose=True
)

# Execute the workflow
result = ai_research_crew.kickoff()

# Print the final blog post
print("\n=== Generated Blog Post ===")
print(result.raw)

┌────────────────────────── Crew Execution Started ───────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: b0f79fcb-e461-4752-b716-66fbd76eeae1                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

🚀 Crew: crew
└── 📋 Task: de9469ec-4cfb-4144-817d-6459274f3d55
    Status: Executing Task...
┌───────────────────────────── 🤖 Agent Started ──────────────────────────────┐
│                                                                             │
│  Agent: AI Researcher     

In [21]:
analysis_task.output

TaskOutput(description='Analyze AI research findings and extract key insights', name=None, expected_output='A structured summary with key takeaways', summary='Analyze AI research findings and extract key insights...', raw='{\n  "insights": [\n    "Generative AI models, such as DALL-E 3 and ChatGPT, have significantly enhanced their capabilities in creating realistic images and engaging in nuanced dialogues, showcasing the advancements in text-to-image and conversational AI.",\n    "The use of self-supervised learning frameworks is revolutionizing AI training methodologies by enabling the development of powerful models with minimal reliance on labeled data, thereby enhancing their performance in various domains like NLP and computer vision.",\n    "Reinforcement learning has made remarkable progress in robotic applications, allowing robots to learn complex tasks through real-time trial and error interactions, which greatly increases their adaptability and operational efficiency.",\n    

In [22]:
analysis_task.output.pydantic

AnalysisSummary(insights=['Generative AI models, such as DALL-E 3 and ChatGPT, have significantly enhanced their capabilities in creating realistic images and engaging in nuanced dialogues, showcasing the advancements in text-to-image and conversational AI.', 'The use of self-supervised learning frameworks is revolutionizing AI training methodologies by enabling the development of powerful models with minimal reliance on labeled data, thereby enhancing their performance in various domains like NLP and computer vision.', 'Reinforcement learning has made remarkable progress in robotic applications, allowing robots to learn complex tasks through real-time trial and error interactions, which greatly increases their adaptability and operational efficiency.', 'AI-driven platforms are transforming drug discovery by enhancing the speed and accuracy of molecular interaction predictions and compound optimization, thereby accelerating the development of new therapeutics and vaccines.', 'The ethic

### **Async Execution**

In [23]:
from pydantic import BaseModel

class AIResearchFindings(BaseModel):
    """Represents structured research on AI breakthroughs."""
    title: str
    key_findings: list[str]

class AIRegulationFindings(BaseModel):
    """Represents structured research on AI regulations."""
    region: str
    key_policies: list[str]

class FinalAIReport(BaseModel):
    """Combines AI research & regulation analysis into a report."""
    executive_summary: str
    key_trends: list[str]

In [24]:
from crewai import Agent

# Researcher for AI breakthroughs
research_agent = Agent(
    role="AI Researcher",
    goal="Find and summarize the latest AI breakthroughs",
    backstory="An expert AI researcher who tracks technological advancements.",
    verbose=True,
    llm=llm
)

# Analyst for AI regulations
regulation_agent = Agent(
    role="AI Policy Analyst",
    goal="Analyze global AI regulations and summarize policies",
    backstory="A government policy expert specializing in AI ethics and laws.",
    verbose=True,
    llm=llm
)

# Writer for the final AI report
writer_agent = Agent(
    role="AI Report Writer",
    goal="Write a structured report combining AI breakthroughs and regulations",
    backstory="A professional technical writer who crafts AI research reports.",
    verbose=True,
    llm=llm
)

In [25]:
from crewai import Task

# Task 1: AI Breakthroughs Research (Asynchronous)
research_ai_task = Task(
    description="Research the latest AI advancements and summarize key breakthroughs.",
    expected_output="A structured list of AI breakthroughs.",
    agent=research_agent,
    output_pydantic=AIResearchFindings,
    async_execution=True  # Runs asynchronously
)

# Task 2: AI Regulation Analysis (Asynchronous)
research_regulation_task = Task(
    description="Analyze the latest AI regulations worldwide and summarize key policies.",
    expected_output="A structured summary of AI regulations by region.",
    agent=regulation_agent,
    output_pydantic=AIRegulationFindings,
    async_execution=True  # Runs asynchronously
)

# Task 3: Generate AI Research Report (Depends on the first two tasks)
generate_report_task = Task(
    description="Write a structured report summarizing AI breakthroughs and regulations.",
    expected_output="A final AI report summarizing both aspects.",
    agent=writer_agent,
    output_pydantic=FinalAIReport,
    context=[research_ai_task, research_regulation_task]  # Waits for these tasks to complete
)

In [26]:
from crewai import Crew

ai_research_crew = Crew(
    agents=[research_agent, regulation_agent, writer_agent],
    tasks=[research_ai_task, research_regulation_task, generate_report_task],
    verbose=True
)

# Execute the workflow
result = ai_research_crew.kickoff()

# Print the final AI report
print("\n=== Generated AI Report ===")
print(result.raw)

┌────────────────────────── Crew Execution Started ───────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: 584af272-0c79-49c7-a867-14e8695fa18f                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

🚀 Crew: crew
├── 📋 Task: cd9a568a-4a77-42c9-a4cd-beacec072142
│   Status: Executing Task...
└── 📋 Task: 4f1fe493-4911-4052-ba58-605efd8f67a0
🚀 Crew: crew
├── 📋 Task: cd9a568a-4a77-42c9-a4cd-beacec072142
│   Status: Executing Task...
└── 📋 Task: 4f1fe493-4911-4052-ba58-605efd8f67

### **Callbacks**

In [27]:
from crewai import Agent

# AI Researcher Agent
research_agent = Agent(
    role="AI News Researcher",
    goal="Find and summarize the latest AI news from trusted sources",
    backstory="You are a dedicated AI journalist who follows the latest advancements in artificial intelligence.",
    verbose=False,
    llm=llm
)

In [28]:
def notify_team(output):

    print(f"""Task Completed!
              Task: {output.description}
              Output Summary: {output.summary}""")

    with open("latest_ai_news.txt", "w") as f:
        f.write(f"Task: {output.description}\n")
        f.write(f"Output Summary: {output.summary}\n")
        f.write(f"Full Output: {output.raw}\n")
    
    print("News summary saved to latest_ai_news.txt")

In [29]:
from crewai import Task

research_news_task = Task(
    description="Find and summarize the latest AI breakthroughs from the last week.",
    expected_output="A structured summary of AI news headlines.",
    agent=research_agent,
    callback=notify_team  # Calls the function after task completion
)

In [30]:
from crewai import Crew

ai_news_crew = Crew(
    agents=[research_agent],
    tasks=[research_news_task],
    verbose=False
)

# Execute the workflow
result = ai_news_crew.kickoff()

Task Completed!
              Task: Find and summarize the latest AI breakthroughs from the last week.
              Output Summary: Find and summarize the latest AI breakthroughs from the last...
News summary saved to latest_ai_news.txt


### **Hierarchial Process**

In [31]:
from crewai import Agent, LLM

# Define the Manager AI
manager_agent = Agent(
    role="Project Research Manager",
    goal="Oversee the project research and ensure timely, high-quality responses.",
    backstory="""An experienced project manager responsible
                 for ensuring project research.""",
    allow_delegation=True,
    verbose=True,
    llm=llm
)

# Define the Technical Support AI
market_demand_agent = Agent(
    role="Market Demand Analyst",
    goal="Write market demand content.",
    backstory="""A skilled market demand analyst who
                 writes market demand content.""",
    allow_delegation=False, 
    verbose=True,
    llm=llm
)

# Define the Fiction Writer AI
risk_analysis_agent = Agent(
    role="Risk Analysis Analyst",
    goal="Write risk analysis content.",
    backstory="""A risk analysis analyst who
                 writes fiction content.""",
    allow_delegation=False, 
    verbose=True,
    llm=llm
)

# Define the Fiction Writer AI
return_on_investment_agent = Agent(
    role="Return on Investment Analyst",
    goal="Write return on investment content.",
    backstory="""A return on investment analyst who
                writes return on investment content.""",
    allow_delegation=False, 
    verbose=True,
    llm=llm
)

In [32]:
from crewai import Task

manager_task = Task(
    description="""Oversee the project research on {project_title} and ensure timely, high-quality responses.""",
    expected_output="A manager-approved response ready to be sent as an article on {project_title}.",
    agent=manager_agent, 
)

market_demand_task = Task(
    description="""Analyze the market demand for the project title '{project_title}'""",
    expected_output="A categorized project title labeled as 'Technical' or 'Fiction'.",
    agent=market_demand_agent, 
)

risk_analysis_task = Task(
    description="""Analyze the risk of the project title '{project_title}'""",
    expected_output="A categorized project title labeled as 'Technical' or 'Fiction'.",
    agent=risk_analysis_agent, 
)

return_on_investment_task = Task(
    description="""Analyze the return on investment of the project title '{project_title}'""",
    expected_output="A categorized project title labeled as 'Technical' or 'Fiction'.",
    agent=return_on_investment_agent, 
)

final_report_task = Task(
    description="""Review the final responses from the 
                   market demand, risk analysis, and return on investment agents
                   and create a final report.""",
    expected_output="""A comprehensive report on the project title '{project_title}'
    containing the market demand, risk analysis, and return on investment.""",
    agent=manager_agent,
)

In [33]:
project_research_crew = Crew(
    agents=[market_demand_agent, risk_analysis_agent, return_on_investment_agent],
    
    tasks=[market_demand_task, risk_analysis_task, return_on_investment_task, final_report_task],
    
    manager_agent=manager_agent,
    
    process=Process.hierarchical,
    
    verbose=True,
)

In [34]:
inputs = {"project_title": "Amazon INC AMZN"}

result = project_research_crew.kickoff(inputs=inputs)

┌────────────────────────── Crew Execution Started ───────────────────────────┐
│                                                                             │
│  Crew Execution Started                                                     │
│  Name: crew                                                                 │
│  ID: 0fef4f63-dc04-4030-a8cf-ea330dd73d97                                   │
│  Tool Args:                                                                 │
│                                                                             │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

🚀 Crew: crew
└── 📋 Task: 57d9aeb8-1287-4188-a0a9-3c920f55ee95
    Status: Executing Task...
┌───────────────────────────── 🤖 Agent Started ──────────────────────────────┐
│                                                                             │
│  Agent: Project Research M

KeyboardInterrupt: 

🚀 Crew: crew
├── 📋 Task: 57d9aeb8-1287-4188-a0a9-3c920f55ee95
│   Assigned to: Project Research Manager
│   Status: ✅ Completed
│   ├── 🔧 Failed Delegate work to coworker (3)
│   ├── 🔧 Failed Delegate work to coworker (6)
│   └── 🔧 Using Delegate work to coworker (7)
├── 📋 Task: f32b63ef-dc24-4594-b910-d6dc243741c1
│   Assigned to: Project Research Manager
│   Status: ✅ Completed
│   ├── 🔧 Failed Delegate work to coworker (10)
│   ├── 🔧 Failed Delegate work to coworker (13)
│   ├── 🔧 Failed Delegate work to coworker (16)
│   ├── 🔧 Failed Delegate work to coworker (19)
│   ├── 🔧 Failed Delegate work to coworker (22)
│   ├── 🔧 Failed Delegate work to coworker (25)
│   ├── 🔧 Failed Delegate work to coworker (28)
│   ├── 🔧 Failed Delegate work to coworker (31)
│   ├── 🔧 Failed Delegate work to coworker (34)
│   └── 🔧 Using Ask question to coworker (1)
├── 📋 Task: ae6fd6a7-3304-4e32-b04d-acdc8c0f05a5
│   Assigned to: Project Research Manager
│   Status: ✅ Completed
│   ├── 🔧 Failed Ask ques